# 05 — Putting It Together

In the late 1980s, bedroom coders across Europe were building bouncing ball demos on home computers — Amiga 500s, Atari STs, Commodore 64s. They were incrementing variables in tight loops, flipping signs at boundaries, multiplying by small numbers to fake friction. They shipped their demos at copyparties and watched other coders crowd around.

They thought they were just making things move on screen.

They were doing applied mathematics.

This notebook is the retrospective. In four notebooks you built a bouncing ball — from a variable that increments to a physically realistic simulation with decay, friction, and wall reflection. Each step added a line of code that was also a classical result in physics and numerical methods. This notebook names every one of those results, gathers the formulas, and makes the full picture explicit.

The Amiga programmer in 1988 and the physicist with their differential equations were doing the same thing.

**Prerequisites:** Notebooks [01 — Linear Motion](./01-linear-motion.ipynb) through [04 — Energy and Decay](./04-energy-and-decay.ipynb). This notebook introduces no new concepts — it names and organises what you already built.

---

## 1. The Origin Story

In 1988, a bedroom programmer in Eindhoven sits at an Amiga 500. He has 512 KB of RAM, a 7 MHz 68000 processor, and a CRT that can do 320×256 in 32 colours. He wants to make a ball bounce across the screen.

He writes something like this:

```python
x, y   = 0, 100
dx, dy = 2, 0

while True:
    dy -= 1                 # pull the ball downward
    y  += dy
    x  += dx

    if y <= 0:              # hit the floor
        dy = -dy * 8 // 10  # bounce, lose some energy

    draw_circle(x, y)
    wait_for_vsync()
```

He isn't thinking about differential equations. He's thinking about whether it *looks* right. He tweaks the gravity number until the drop feels realistic. He tweaks the bounce multiplier until the decay feels satisfying.

He has just implemented:

- **Forward Euler integration** of a second-order ordinary differential equation
- **Elastic reflection** with a **coefficient of restitution**
- **Exponential decay** via a **geometric sequence**
- **Boundary conditions** on a **piecewise function**

A physics professor would have written the same system as $\frac{d^2 y}{dt^2} = g$ with boundary conditions, chosen a numerical integrator, and spent three lectures on it.

Both produce the same code. The professor has the vocabulary. The bedroom coder had the intuition. This notebook gives you both.

---

## 2. The Complete Demo

The entire simulation in one cell. No animation library, no helper functions — just the physics loop. Read it top to bottom. It should take about thirty seconds.

Every line is annotated in the next section.

In [ ]:
x,  y  = 10.0, 26.0
dx, dy = 1.5,  0.0

gravity     = -0.15
restitution = 0.82
friction    = 0.99
floor       = 4.0
left, right = 0.0, 118.0
clamp       = 0.4

positions = []
for frame in range(300):
    dy += gravity
    y  += dy
    x  += dx
    dx *= friction

    if y <= floor:
        y  = floor
        dy = -dy * restitution
        if abs(dy) < clamp:
            dy = 0.0
    if x >= right or x <= left:
        dx = -dx

    positions.append((x, y, dx, dy))

rest_frame = next((i for i, p in enumerate(positions) if p[3] == 0.0), None)
print(f'Frames simulated: {len(positions)}')
print(f'Ball settled at frame: {rest_frame}')
print(f'Final position: x={positions[-1][0]:.1f}, y={positions[-1][1]:.1f}')

---

## 3. The Annotated Version

The same loop, line by line. Each operation has a name.

```python
dy += gravity           # [1]
y  += dy                # [2]
x  += dx                # [3]
dx *= friction          # [4]

if y <= floor:          # [5]
    y  = floor
    dy = -dy * restitution   # [6]
    if abs(dy) < clamp:
        dy = 0.0        # [7]
if x >= right or x <= left:  # [8]
    dx = -dx            # [9]
```

| # | Code | Plain English | Mathematical name | Formula |
|---|------|---------------|-------------------|---------|
| 1 | `dy += gravity` | Vertical speed increases by a constant each frame | **Newton's 2nd law** — constant acceleration | $v_y \leftarrow v_y + g \Delta t$ |
| 2 | `y += dy` | Vertical position updates from current velocity | **Forward Euler integration** | $y \leftarrow y + v_y \Delta t$ |
| 3 | `x += dx` | Horizontal position updates from current velocity | **Linear motion** | $x \leftarrow x + v_x \Delta t$ |
| 4 | `dx *= friction` | Horizontal speed multiplied by a fraction $< 1$ each frame | **Exponential decay** (continuous) | $v_x \leftarrow \mu \cdot v_x$ |
| 5 | `if y <= floor:` | Ball has reached the lower domain boundary | **Boundary condition** | $y = y_\text{floor}$ |
| 6 | `dy = -dy * restitution` | Velocity reverses and loses energy at impact | **Elastic reflection with dissipation** | $v_y \leftarrow -e \cdot v_y$ |
| 7 | `if abs(dy) < clamp: dy = 0` | Stop below threshold — Zeno's series has converged | **Numerical termination criterion** | $|v_y| < \varepsilon \Rightarrow v_y = 0$ |
| 8 | `if x >= right or x <= left:` | Ball has reached a lateral boundary | **Boundary condition** (lateral walls) | $x = x_R$ or $x = x_L$ |
| 9 | `dx = -dx` | Horizontal velocity reverses at the wall | **Elastic reflection** | $v_x \leftarrow -v_x$ |

Nine operations. Nine classical names. Written in a loop on a home computer.

---

## 4. The Complete Mathematical Model

Gathered together, the equations form a complete dynamical system. This is what a physicist would write down before touching any code.

**Horizontal motion** — linear position with exponentially decaying velocity:

$$x(t) = x_0 + v_x t \qquad v_x(t) = v_{x,0} \cdot \mu^t$$

**Vertical motion between bounces** — parabolic arc:

$$y(t) = y_0 + v_0\,t + \tfrac{1}{2}g\,t^2$$

Derived in notebook 02: the $\tfrac{1}{2}$ comes from the triangular number formula — the accumulated sum of discrete velocities.

**At each vertical bounce** — reflection with dissipation:

$$v_y \;\longrightarrow\; -e \cdot v_y$$

Applied once at impact. $e = 1$ is notebook 03 (no energy loss). $e = 0$ stops the ball dead.

**Decay envelope** — peak height after $n$ bounces:

$$h_n = h_0 \cdot e^{2n}$$

Derived in notebook 04: peak height is proportional to $v^2$, and $v$ decays by $e$ per bounce, so height decays by $e^2$.

**At each wall bounce** — elastic reflection:

$$v_x \;\longrightarrow\; -v_x$$

---

Assembled as a system with boundary conditions and dissipation:

$$\frac{d^2 y}{dt^2} = g \qquad \frac{d^2 x}{dt^2} = 0$$

$$v_y \to -e \cdot v_y \quad \text{at} \quad y = y_\text{floor} \qquad v_x \to -v_x \quad \text{at} \quad x \in \{x_L, x_R\}$$

$$v_x \to \mu \cdot v_x \quad \text{per timestep} \qquad |v_y| < \varepsilon \Rightarrow v_y = 0$$

This is not a collection of tricks. It is one physical system described precisely.

In [ ]:
import math

# Parameters matching the simulation above
h0     = 26.0 - 4.0   # height above floor (26 - 4 = 22)
g_abs  = 0.15
e_r    = 0.82
mu     = 0.99
dx0    = 1.5

# Free fall: exact impact time and velocity
t_fall   = math.sqrt(2 * h0 / g_abs)
v_impact = math.sqrt(2 * g_abs * h0)
print(f'Free fall from h = {h0}  (|g| = {g_abs}):')
print(f'  Impact time:     {t_fall:.2f} frames    (sqrt(2h/|g|))')
print(f'  Impact velocity: {v_impact:.4f}       (sqrt(2|g|h))')
print()

# Peak heights: h_n = h0 * e^(2n)
print(f'Peak heights  h_n = h0 x e^(2n)  (h0={h0}, e={e_r}, e^2={e_r**2:.4f}):')
print(f'{"n":>4}  {"h_n":>10}  {"% of h0":>10}  {"ratio to prev":>15}')
prev = h0
for n in range(7):
    hn    = h0 * (e_r ** (2 * n))
    ratio = hn / prev if n > 0 else float('nan')
    flag  = f'  <- e^2 = {e_r**2:.4f}' if n == 1 else ''
    print(f'{n:>4}  {hn:>10.4f}  {100*hn/h0:>9.2f}%  {ratio:>15.4f}{flag}')
    prev  = hn
print()

# Horizontal decay
print(f'Horizontal speed decay  (dx0={dx0}, friction={mu}):')
for t in [0, 50, 100, 200, 300]:
    vx = dx0 * mu**t
    print(f'  t={t:>4}: dx = {vx:.5f}')
t_half = math.log(0.5) / math.log(mu)
print(f'  Halves every {t_half:.0f} frames  (log(0.5)/log({mu}))')
print()

# Zeno total bounce time
T_total = (v_impact / g_abs) * (1 + e_r) / (1 - e_r)
print(f'Zeno total bounce duration:  T = v0/|g| x (1+e)/(1-e)')
print(f'  = {v_impact:.4f}/{g_abs} x {(1+e_r)/(1-e_r):.3f} = {T_total:.1f} frames')
print(f'  (Clamping at |dy| < 0.4 stops the simulation earlier)')

---

## 5. Forward Euler — and Gradient Descent

Every line of the physics loop is one step of a numerical integrator applied to a differential equation.

The equation governing the ball's vertical motion is Newton's second law:

$$\frac{d^2 y}{dt^2} = g$$

To simulate it numerically, introduce velocity as a separate variable and rewrite as a first-order system:

$$\frac{dv_y}{dt} = g \qquad \frac{dy}{dt} = v_y$$

**Forward Euler** discretises this with timestep $\Delta t$:

$$v_y^{n+1} = v_y^n + g \cdot \Delta t$$
$$y^{n+1} = y^n + v_y^n \cdot \Delta t$$

This is the game loop with $\Delta t = 1$:

```python
dy += gravity   # v_y^{n+1} = v_y^n + g * dt
y  += dy        # y^{n+1}   = y^n  + v_y^n * dt  ← uses old velocity (Forward Euler)
```

The complete system has four named parts:

| Part | Mathematical name | Code |
|------|------------------|------|
| $\frac{d^2y}{dt^2} = g$ | **ODE** — Newton's second law | `dy += gravity; y += dy` |
| $v_y \to -e \cdot v_y$ at $y = y_\text{floor}$ | **Boundary condition** — dissipative reflection | `if y <= floor: dy = -dy * e` |
| $v_x \to \mu \cdot v_x$ each frame | **Dissipation** — velocity-proportional damping | `dx *= friction` |
| $\Delta t = 1$ explicit step | **Forward Euler integrator** | the loop body |

---

Forward Euler has a famous cousin: **gradient descent**.

```python
# Physics loop              # ML optimisation
dy += gravity               grad  = compute_gradient(params)
y  += dy                    params -= lr * grad
```

Same update structure. Same numerical integrator. In physics, the force is gravity — it determines the direction of acceleration. In ML, the force is the gradient — it determines the direction of parameter update.

The ball follows the curve of a potential well. The network follows the curve of the loss surface. Both use Forward Euler to take one step at a time.

You have been running gradient descent since notebook 01.

---

## 6. The Full Demo

The simulation rendered: floor bounces, wall bounces, horizontal friction, restitution, and velocity clamping. Below the animation, three diagnostic plots show the complete motion history.

In [ ]:
# FuncAnimation requires an interactive backend — add '%matplotlib widget' above if it renders static
import matplotlib.pyplot as plt
import matplotlib.animation as animation

x0, y0      = 10.0, 26.0
dx0, dy0    = 1.5,  0.0
gravity     = -0.15
restitution = 0.82
friction    = 0.99
floor_y     = 4.0
left_x      = 0.0
right_x     = 118.0
clamp_v     = 0.4
FRAMES      = 300

# Pre-simulate the full trajectory
x, y, dy, dx = x0, y0, dy0, dx0
all_positions = []
bounce_events = []   # (x_pos, frame, 'floor'|'wall')
stopped       = False

for f in range(FRAMES):
    if not stopped:
        dy += gravity
        y  += dy

    dx *= friction
    x  += dx

    if not stopped and y <= floor_y:
        y  = floor_y
        dy = -dy * restitution
        bounce_events.append((x, f, 'floor'))
        if abs(dy) < clamp_v:
            dy      = 0.0
            stopped = True

    if x <= left_x:
        x  = left_x
        dx = -dx
        bounce_events.append((x, f, 'wall'))
    elif x >= right_x:
        x  = right_x
        dx = -dx
        bounce_events.append((x, f, 'wall'))

    all_positions.append((x, y, dx, dy))

# ── Build figure ──
fig, ax = plt.subplots(figsize=(10, 3))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, 120)
ax.set_ylim(0, 30)
ax.set_aspect('equal')
ax.axis('off')

# Floor and walls
ax.axhline(floor_y, color='#45475a', linewidth=1.5)
ax.axvline(left_x  + 0.4, color='#45475a', linewidth=1.5)
ax.axvline(right_x - 0.4, color='#45475a', linewidth=1.5)
ax.text(1, floor_y + 0.7, 'floor', color='#6c7086', fontsize=8, fontfamily='monospace')

# Static floor-bounce markers
floor_bounces = [(bx, bf) for bx, bf, bt in bounce_events if bt == 'floor']
for i, (bx, _) in enumerate(floor_bounces, 1):
    ax.plot(bx, floor_y, 'v', color='#f38ba8', markersize=6, zorder=5)
    ax.text(bx, floor_y - 2.5, f'#{i}', color='#f38ba8',
            fontsize=7, ha='center', fontfamily='monospace')

ball = plt.Circle((x0, y0), radius=3, color='#89dceb')
ax.add_patch(ball)
trail_dots = [plt.Circle((0, 0), radius=0.8, color='#89dceb', alpha=0) for _ in range(22)]
for dot in trail_dots:
    ax.add_patch(dot)

info = ax.text(2, 28.5, '', color='#cdd6f4', fontsize=9, fontfamily='monospace')

def update(frame):
    px, py, pdx, pdy = all_positions[frame]
    ball.set_center((px, py))
    for i, dot in enumerate(trail_dots):
        idx = frame - len(trail_dots) + i + 1
        if 0 <= idx < len(all_positions):
            dot.set_center(all_positions[idx][:2])
            dot.set_alpha(0.04 + 0.045 * i)
        else:
            dot.set_alpha(0)
    info.set_text(f't={frame:>3}   dy={pdy:+.2f}   dx={pdx:.2f}')
    return [ball, info] + trail_dots

ani = animation.FuncAnimation(fig, update, frames=FRAMES, interval=60, blit=True)
plt.tight_layout()
plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt

ts     = list(range(len(all_positions)))
xs     = [p[0] for p in all_positions]
ys     = [p[1] for p in all_positions]
dxs    = [p[2] for p in all_positions]
dys    = [p[3] for p in all_positions]
speeds = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(dxs, dys)]

floor_bounces = [(bf, all_positions[bf]) for _, bf, bt in bounce_events if bt == 'floor']
wall_bounces  = [(bf, all_positions[bf]) for _, bf, bt in bounce_events if bt == 'wall']

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
fig.patch.set_facecolor('#1e1e2e')

def style_ax(ax, title, xlabel, ylabel):
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='#cdd6f4', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#45475a')
    ax.set_title(title, color='#cdd6f4', pad=8, fontsize=10)
    ax.set_xlabel(xlabel, color='#cdd6f4', fontsize=9)
    ax.set_ylabel(ylabel, color='#cdd6f4', fontsize=9)

# — Plot 1: x position over time
style_ax(axes[0], 'x position  (linear + wall reflections)', 't (frame)', 'x')
axes[0].plot(ts, xs, color='#89dceb', linewidth=1.5)
axes[0].axhline(left_x,  color='#45475a', linewidth=0.8, linestyle='--', alpha=0.6)
axes[0].axhline(right_x, color='#45475a', linewidth=0.8, linestyle='--', alpha=0.6)
for bf, _ in wall_bounces:
    axes[0].axvline(bf, color='#fab387', linewidth=0.8, alpha=0.5)
axes[0].text(5, right_x - 8, 'right wall', color='#6c7086', fontsize=8)
axes[0].text(5, left_x  + 3, 'left wall',  color='#6c7086', fontsize=8)

# — Plot 2: y position over time
style_ax(axes[1], 'y position  (piecewise quadratic, decaying)', 't (frame)', 'y')
axes[1].plot(ts, ys, color='#a6e3a1', linewidth=1.5)
axes[1].axhline(floor_y, color='#45475a', linewidth=0.8, linestyle='--', alpha=0.6)
for bf, _ in floor_bounces:
    axes[1].plot(bf, floor_y, 'v', color='#f38ba8', markersize=5, zorder=5)
axes[1].text(5, floor_y + 0.8, 'floor', color='#6c7086', fontsize=8)

# — Plot 3: speed over time
style_ax(axes[2], 'speed  (exponential decay envelope)', 't (frame)', '|velocity|')
axes[2].plot(ts, speeds, color='#fab387', linewidth=1.2, alpha=0.7, label='speed')
if floor_bounces:
    env_t = [bf for bf, _ in floor_bounces]
    env_s = [speeds[bf] for bf, _ in floor_bounces]
    axes[2].plot(env_t, env_s, 'o--', color='#f38ba8', markersize=6,
                 linewidth=1.2, label='speed at each bounce')
axes[2].legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.85, fontsize=8)

plt.tight_layout()
plt.show()

---

## What You Actually Learned

Each notebook in this series introduced one concept. Here is the full map: what you built, what it is called, and where it connects.

| Notebook | Concept | Forward connection |
|----------|---------|-------------------|
| 01 — Linear Motion | **Linear equations** — constant rate of change | Derivatives: the slope of a curve at a point |
| 02 — Acceleration | **Quadratic equations** — accumulating change | Integration: the $\frac{1}{2}$ is a Riemann sum |
| 03 — Bouncing | **Piecewise functions**, boundary conditions | PDEs: heat, wave, and Schrödinger equations all have boundary conditions |
| 04 — Energy and Decay | **Exponential decay** — geometric sequences | Weight decay in ML, signal attenuation, half-life |
| All notebooks | **Forward Euler** — first-order integrator | Gradient descent, Runge-Kutta, all ODE solvers |

The connections are not distant:

- **Linear motion → calculus.** The derivative is instantaneous velocity — the limit of $(x_{n+1} - x_n) / \Delta t$ as $\Delta t \to 0$. You computed the finite version every frame.
- **Acceleration → integration.** The $\tfrac{1}{2}gt^2$ formula comes from summing accumulating velocities — a discrete Riemann sum. Definite integration is the continuous limit of exactly that process.
- **Bouncing → PDEs.** Every partial differential equation simulation has boundary conditions — constraints on what happens at the edges of the domain. Ours was `if y <= floor: dy = -dy`. Heat equations, wave equations, fluid simulations: same structure, different equations.
- **Exponential decay → everywhere.** The same $A \cdot r^n$ pattern appears in radioactive decay, compound interest, ML weight regularisation, RC circuit discharge, and acoustic reverberation. The bouncing ball is one instance of a pattern that runs through all of quantitative science.
- **Forward Euler → gradient descent.** The loop that updates `dy` and `y` every frame updates parameters and gradients every step in an optimiser. The ball settles into the bottom of a potential well. The model settles into the bottom of the loss surface.

---

You built this for fun. It turns out fun and mathematics have always been the same thing. The bedroom programmer who tweaked the bounce multiplier until it felt right, the physicist who solved the ODE analytically, and the ML engineer who tunes the learning rate — all working the same problem from different directions. The game loop was always the math. It just needed the names.